# Komparasi Model Klasifikasi Kualitas Udara

Bandingkan 3 model klasifikasi:
1. **Random Forest** (baseline)
2. **XGBoost** (gradient boosting)
3. **LightGBM** (gradient boosting, lebih cepat)

Dataset: sama dengan `train_classification.ipynb` (60% KLHK + 15% sensor + 25% sintetis)
Class weight: balanced semua model

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
import joblib, os, time
from supabase import create_client, Client
from datetime import datetime, timedelta

import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')

print("Setup done")

### Load Dataset (sama dengan train_classification.ipynb)

In [ ]:
SUPABASE_URL = os.getenv("SUPABASE_URL", "")
SUPABASE_KEY = os.getenv("SUPABASE_KEY", "")

if not SUPABASE_URL or not SUPABASE_KEY:
    env_path = ".env"
    if os.path.exists(env_path):
        for line in open(env_path):
            if "=" in line and not line.startswith("#"):
                k, _, v = line.partition("=")
                os.environ[k.strip()] = v.strip().strip(chr(34))
    SUPABASE_URL = os.getenv("SUPABASE_URL", "")
    SUPABASE_KEY = os.getenv("SUPABASE_KEY", "")

SUPABASE_URL = SUPABASE_URL.replace("http://", "https://")
supabase: Client = create_client(SUPABASE_URL, SUPABASE_KEY)

KLHK_DIR = "../klhk/"
FILES = [
    "Filedata Indeks Standar Pencemaran Udara (ISPU) Tahun 2011.csv",
    "Filedata Indeks Standar Pencemaran Udara (ISPU) Tahun 2012.csv",
    "Filedata Indeks Standar Pencemaran Udara (ISPU) Tahun 2013.csv",
    "Filedata Indeks Standar Pencemaran Udara (ISPU) Tahun 2015.csv",
    "Filedata Data Indeks Standar Pencemaran Udara DKI Jakarta Tahun 2017.csv",
    "Filedata Indeks Standar Pencemaran Udara (ISPU) Tahun 2019.csv",
    "Filedata Indeks Standar Pencemaran Udara (ISPU) Tahun 2020.csv",
    "Filedata Indeks Standar Pencemaran Udara (ISPU) Tahun 2022.csv",
]

all_dfs = []
for f in FILES:
    path = os.path.join(KLHK_DIR, f)
    if not os.path.exists(path):
        print(f"SKIP: {f}")
        continue
    all_dfs.append(pd.read_csv(path))

raw = pd.concat(all_dfs, ignore_index=True)
print(f"KLHK raw: {len(raw)} baris")

LABEL_MAP = {
    "BAIK": "Baik", "SEDANG": "Sedang", "TIDAK SEHAT": "Tidak Sehat",
    "SANGAT TIDAK SEHAT": "Sangat Tidak Sehat", "BERBAHAYA": "Berbahaya",
}

def ispi_to_conc(ispi, bp):
    if ispi <= 0: return 0
    for cl, ch, il, ih in bp:
        if ispi <= ih: return cl + (ispi - il) / (ih - il) * (ch - cl)
    return bp[-1][1]

BP_PM25 = [(0,15,0,50),(15,35,50,100),(35,55,100,200),(55,150,200,300),(150,250,300,400),(250,350,400,500)]
BP_PM10 = [(0,50,0,50),(50,150,50,100),(150,350,100,200),(350,420,200,300),(420,500,300,400),(500,600,400,500)]
BP_CO   = [(0,5000,0,50),(5000,10000,50,100),(10000,17000,100,200),(17000,34000,200,300),(34000,46000,300,400),(46000,56000,400,500)]
BP_NO2  = [(0,40,0,50),(40,80,50,100),(80,180,100,200),(180,280,200,300),(280,565,300,400),(565,665,400,500)]
BP_O3   = [(0,60,0,50),(60,120,50,100),(120,180,100,200),(180,240,200,300),(240,400,300,500)]
O3_CONV = 1.963

klhk_rows = []
for _, r in raw.iterrows():
    cat_col = "kategori" if "kategori" in raw.columns and "categori" not in raw.columns else "categori"
    if cat_col not in r or pd.isna(r.get(cat_col)): continue
    label = LABEL_MAP.get(str(r[cat_col]).strip().upper())
    if not label: continue
    pm10_ispi = pd.to_numeric(r.get("pm10", r.get("pm_10", 0)), errors="coerce")
    co_ispi   = pd.to_numeric(r.get("co", 0), errors="coerce")
    o3_ispi   = pd.to_numeric(r.get("o3", 0), errors="coerce")
    no2_ispi  = pd.to_numeric(r.get("no2", 0), errors="coerce")
    pm25_ispi = pd.to_numeric(r.get("pm_duakomalima"), errors="coerce") if pd.notna(r.get("pm_duakomalima")) else pm10_ispi * 0.7
    pm25 = ispi_to_conc(pm25_ispi if pd.notna(pm25_ispi) else 0, BP_PM25)
    pm10 = ispi_to_conc(pm10_ispi if pd.notna(pm10_ispi) else 0, BP_PM10)
    co   = ispi_to_conc(co_ispi if pd.notna(co_ispi) else 0, BP_CO)
    no2  = ispi_to_conc(no2_ispi if pd.notna(no2_ispi) else 0, BP_NO2)
    o3_ppb = ispi_to_conc(o3_ispi if pd.notna(o3_ispi) else 0, BP_O3)
    o3 = o3_ppb * O3_CONV
    if pm25 > 0 and pm10 > 0:
        klhk_rows.append({"PM2.5": pm25, "PM10": pm10, "CO": co, "NO2": no2, "O3": o3, "Label": label, "src": "klhk"})

df_klhk = pd.DataFrame(klhk_rows)
print(f"KLHK clean: {len(df_klhk)} baris")

In [ ]:
TABLE = "tb_konsentrasi_gas"
since = (datetime.utcnow() - timedelta(days=90)).isoformat()
all_data, offset = [], 0
while True:
    resp = supabase.table(TABLE) \
        .select("pm25_ugm3,pm10_ugm3,co_ugm3,no2_ugm3,o3_ugm3") \
        .gte("created_at", since).order("created_at", desc=False).range(offset, offset+999).execute()
    if not resp.data: break
    all_data.extend(resp.data)
    offset += len(resp.data)

df_sensor = pd.DataFrame(all_data)
for c in ["pm25_ugm3","pm10_ugm3","co_ugm3","no2_ugm3","o3_ugm3"]:
    df_sensor[c] = pd.to_numeric(df_sensor[c], errors="coerce")
df_sensor = df_sensor.dropna(subset=["pm25_ugm3","pm10_ugm3","o3_ugm3"])

def conc_to_ispi(val, bp):
    if val <= 0: return 0
    for cl, ch, il, ih in bp:
        if val <= ch: return il + (val - cl) / (ch - cl) * (ih - il)
    return bp[-1][3]

def ispi_to_label(ispi):
    if ispi <= 50: return "Baik"
    if ispi <= 100: return "Sedang"
    if ispi <= 200: return "Tidak Sehat"
    if ispi <= 300: return "Sangat Tidak Sehat"
    return "Berbahaya"

sensor_rows = []
for _, r in df_sensor.iterrows():
    pm25, pm10, co, no2 = r["pm25_ugm3"], r["pm10_ugm3"], r["co_ugm3"], r["no2_ugm3"]
    o3_ugm3 = r["o3_ugm3"]
    o3_ppb = o3_ugm3 / O3_CONV
    ispis = [conc_to_ispi(pm25,BP_PM25), conc_to_ispi(pm10,BP_PM10), conc_to_ispi(co,BP_CO), conc_to_ispi(no2,BP_NO2), conc_to_ispi(o3_ppb,BP_O3)]
    label = ispi_to_label(max(ispis))
    sensor_rows.append({"PM2.5": pm25, "PM10": pm10, "CO": co, "NO2": no2, "O3": o3_ugm3, "Label": label, "src": "sensor"})

df_sensor_clean = pd.DataFrame(sensor_rows)
print(f"Sensor clean: {len(df_sensor_clean)} baris")

In [ ]:
np.random.seed(42)

SYNTH = {
    "Baik": {"PM2.5": (1,14), "PM10": (5,48), "CO": (200,4800), "NO2": (1,38), "O3": (2,55*O3_CONV)},
    "Sedang": {"PM2.5": (16,34), "PM10": (51,148), "CO": (5100,9800), "NO2": (41,78), "O3": (61*O3_CONV,119*O3_CONV)},
    "Tidak Sehat": {"PM2.5": (36,54), "PM10": (151,348), "CO": (10100,16800), "NO2": (81,178), "O3": (121*O3_CONV,179*O3_CONV)},
    "Sangat Tidak Sehat": {"PM2.5": (56,149), "PM10": (351,419), "CO": (17100,33800), "NO2": (181,279), "O3": (181*O3_CONV,239*O3_CONV)},
    "Berbahaya": {"PM2.5": (151,350), "PM10": (421,550), "CO": (34100,50000), "NO2": (281,500), "O3": (241*O3_CONV,380*O3_CONV)},
}

synth_rows = []
for label, ranges in SYNTH.items():
    n = 800 if label in ("Baik", "Berbahaya", "Sangat Tidak Sehat") else 300
    for _ in range(n):
        synth_rows.append({
            "PM2.5": np.random.uniform(*ranges["PM2.5"]),
            "PM10": np.random.uniform(*ranges["PM10"]),
            "CO": np.random.uniform(*ranges["CO"]),
            "NO2": np.random.uniform(*ranges["NO2"]),
            "O3": np.random.uniform(*ranges["O3"]),
            "Label": label, "src": "synthetic"
        })

df_synth = pd.DataFrame(synth_rows)
print(f"Sintetis: {len(df_synth)} baris")

In [ ]:
df_all = pd.concat([df_klhk, df_sensor_clean, df_synth], ignore_index=True)
FEATURES = ["PM2.5", "PM10", "CO", "NO2", "O3"]
LABELS = ["Baik", "Sedang", "Tidak Sehat", "Sangat Tidak Sehat", "Berbahaya"]

X = df_all[FEATURES]
y = df_all["Label"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

print(f"Dataset: {len(df_all)} baris")
print(f"Train: {len(X_train)}, Test: {len(X_test)}")
print(f"\nDistribusi label:")
print(y.value_counts().to_string())

In [ ]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
le.fit(LABELS)
y_train_enc = le.transform(y_train)
y_test_enc = le.transform(y_test)

RF = RandomForestClassifier(n_estimators=100, random_state=42, class_weight="balanced", n_jobs=-1)
XGB = XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.1, random_state=42, eval_metric="mlogloss", verbosity=0)
LGBM = __import__("lightgbm", fromlist=["LGBMClassifier"]).LGBMClassifier(n_estimators=100, max_depth=6, learning_rate=0.1, random_state=42, class_weight="balanced", verbosity=-1, n_jobs=-1)

MODELS = {"Random Forest": RF, "XGBoost": XGB, "LightGBM": LGBM}

print("Model defined (labels encoded):", list(le.classes_))

### Train & Evaluate Semua Model

In [ ]:
results = []
trained_models = {}

for name, model in MODELS.items():
    print(f"\n{'='*50}")
    print(f"Training {name}...")
    print('='*50)

    t0 = time.time()
    if name in ("XGBoost", "LightGBM"):
        model.fit(X_train, y_train_enc)
        y_pred = le.inverse_transform(model.predict(X_test))
        y_proba = model.predict_proba(X_test)
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_proba = model.predict_proba(X_test)
    train_time = time.time() - t0

    acc = accuracy_score(y_test, y_pred)
    prec_w = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    rec_w  = recall_score(y_test, y_pred, average='weighted', zero_division=0)
    f1_w   = f1_score(y_test, y_pred, average='weighted', zero_division=0)
    prec_m = precision_score(y_test, y_pred, average='macro', zero_division=0)
    rec_m  = recall_score(y_test, y_pred, average='macro', zero_division=0)
    f1_m   = f1_score(y_test, y_pred, average='macro', zero_division=0)

    cm = confusion_matrix(y_test, y_pred, labels=LABELS)

    results.append({
        "Model": name,
        "Accuracy": round(acc*100, 2),
        "Precision_W": round(prec_w*100, 2),
        "Recall_W": round(rec_w*100, 2),
        "F1_W": round(f1_w*100, 2),
        "Precision_M": round(prec_m*100, 2),
        "Recall_M": round(rec_m*100, 2),
        "F1_M": round(f1_m*100, 2),
        "Train_time": round(train_time, 3),
        "CM": cm,
    })

    trained_models[name] = model

    print(f"  Accuracy:  {acc*100:.2f}%")
    print(f"  Precision (weighted): {prec_w*100:.2f}%")
    print(f"  Recall (weighted):    {rec_w*100:.2f}%")
    print(f"  F1 (weighted):       {f1_w*100:.2f}%")
    print(f"  F1 (macro):          {f1_m*100:.2f}%")
    print(f"  Train time:          {train_time:.3f}s")
    print(f"\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=LABELS, zero_division=0))

df_results = pd.DataFrame(results).drop(columns=['CM'])
print("\n\n" + "="*60)
print("RINGKASAN PERBANDINGAN MODEL")
print("="*60)
print(df_results.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

metrics_w = ['Accuracy', 'Precision_W', 'Recall_W', 'F1_W']
metrics_m = ['Precision_M', 'Recall_M', 'F1_M', 'Train_time']

x = np.arange(len(MODELS))
width = 0.18

colors = ['#6366f1', '#f97316', '#10b981', '#ef4444']

for i, metric in enumerate(metrics_w):
    vals = [r[metric] for r in results]
    axes[0].bar(x + i*width, vals, width, label=metric.replace('_W',''), color=colors[i], alpha=0.85)

axes[0].set_xticks(x + 1.5*width)
axes[0].set_xticklabels(list(MODELS.keys()), fontsize=11)
axes[0].set_ylabel('Score (%)')
axes[0].set_title('Weighted Metrics', fontsize=13, fontweight='bold')
axes[0].set_ylim(0, 110)
axes[0].legend(fontsize=9, loc='lower right')
axes[0].axhline(y=90, color='gray', linestyle='--', alpha=0.5, linewidth=0.8)

for i, metric in enumerate(metrics_m):
    vals = [r[metric] for r in results]
    axes[1].bar(x + i*width, vals, width, label=metric.replace('_M',''), color=colors[i], alpha=0.85)

axes[1].set_xticks(x + 1.5*width)
axes[1].set_xticklabels(list(MODELS.keys()), fontsize=11)
axes[1].set_ylabel('Score / Waktu')
axes[1].set_title('Macro Metrics + Training Time', fontsize=13, fontweight='bold')
axes[1].legend(fontsize=9, loc='lower right')
axes[1].axhline(y=90, color='gray', linestyle='--', alpha=0.5, linewidth=0.8)

plt.suptitle('Perbandingan 3 Model Klasifikasi Kualitas Udara', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: model_comparison.png")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for i, r in enumerate(results):
    sns.heatmap(r['CM'], annot=True, fmt='d', cmap='Blues', ax=axes[i],
                xticklabels=LABELS, yticklabels=LABELS,
                annot_kws={'size': 9}, cbar=False)
    axes[i].set_title(f"{r['Model']}\nAcc: {r['Accuracy']:.1f}%", fontsize=11, fontweight='bold')
    axes[i].set_xlabel('Predicted', fontsize=9)
    axes[i].set_ylabel('Actual', fontsize=9)
    axes[i].tick_params(axis='x', rotation=30)

plt.suptitle('Confusion Matrix per Model', fontsize=14, fontweight='bold', y=1.05)
plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: confusion_matrices.png")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for i, name in enumerate(MODELS.keys()):
    model = trained_models[name]
    if hasattr(model, 'feature_importances_'):
        imp = pd.DataFrame({'Feature': FEATURES, 'Importance': model.feature_importances_})
        imp = imp.sort_values('Importance', ascending=True)
        axes[i].barh(imp['Feature'], imp['Importance'], color=['#6366f1','#f97316','#10b981','#ef4444','#8b5cf6'][i % 5], alpha=0.85)
    else:
        axes[i].text(0.5, 0.5, 'N/A', ha='center', va='center', fontsize=14, transform=axes[i].transAxes)
    axes[i].set_title(f'{name}', fontsize=11, fontweight='bold')
    axes[i].set_xlabel('Importance', fontsize=9)

plt.suptitle('Feature Importance per Model', fontsize=14, fontweight='bold', y=1.05)
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: feature_importance.png")

In [ ]:
best_acc = max(results, key=lambda r: r['Accuracy'])
best_f1_w = max(results, key=lambda r: r['F1_W'])
best_f1_m = max(results, key=lambda r: r['F1_M'])
fastest = min(results, key=lambda r: r['Train_time'])

print("="*60)
print("REKOMENDASI MODEL")
print("="*60)
print(f"\n  Akurasi Tertinggi : {best_acc['Model']} ({best_acc['Accuracy']:.2f}%)")
print(f"  F1 Weighted Tertinggi: {best_f1_w['Model']} ({best_f1_w['F1_W']:.2f}%)")
print(f"  F1 Macro Tertinggi: {best_f1_m['Model']} ({best_f1_m['F1_M']:.2f}%)")
print(f"  Training Tercepat: {fastest['Model']} ({fastest['Train_time']:.3f}s)")

winner = best_f1_w['Model']
print(f"\n  REKOMENDASI AKHIR: {winner}")
print(f"  Alasan: F1 Weighted tertinggi ({best_f1_w['F1_W']:.2f}%) dan")
print(f"           akurasi baik ({best_acc['Accuracy']:.2f}%)")

In [ ]:
best_model_name = winner
best_model = trained_models[best_model_name]

joblib.dump(best_model, f"air_quality_classifier_{best_model_name.replace(' ','_').lower()}.pkl")
print(f"Best model saved: air_quality_classifier_{best_model_name.replace(' ','_').lower()}.pkl")

import json
summary = {
    "best_model": best_model_name,
    "accuracy": best_acc['Accuracy'],
    "f1_weighted": best_f1_w['F1_W'],
    "f1_macro": best_f1_m['F1_M'],
    "all_results": [
        {k: (v.tolist() if hasattr(v,'tolist') else v) for k,v in r.items() if k != 'CM'}
        for r in results
    ]
}
with open("classification_comparison.json", "w") as f:
    json.dump(summary, f, indent=2)
print("Summary saved: classification_comparison.json")